# Segger Checkpoint and Segmentation Workflows


## Requirements

- Segger installed (with GPU dependencies if running segmentation)
- A dataset in raw platform format or SpatialData Zarr
- A Lightning checkpoint (.ckpt) if using checkpoint modes


## Scenario 1: Standard segmentation (train + predict)

```bash
segger segment -i /path/to/data -o /path/to/out
```

Notes:
- This trains a new model and then runs prediction.
- Use `--output-format` to write merged, spatialdata, or anndata outputs.


## Scenario 2: Predict-only from a checkpoint

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode predict
```

Notes:
- Skips training and only runs prediction.
- Uses the checkpoint model weights as-is.


## Scenario 3: Resume training from a checkpoint

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode resume \
  --n-epochs 20
```

Notes:
- Restores optimizer state and resumes training.
- Requires identical gene vocabulary (`vocab_mode` is strict).


## Scenario 4: Finetune from a checkpoint

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --learning-rate 1e-4 \
  --n-epochs 5
```

Notes:
- Loads weights but starts a fresh optimizer.
- Best practice: small learning rate and short finetune.


## Scenario 5: Vocab overlap finetune

If your current gene list differs from the checkpoint, use overlap mode.

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --vocab-mode overlap \
  --checkpoint-vocab /path/to/checkpoint_genes.txt
```

Notes:
- `checkpoint_genes.txt` is one gene per line in checkpoint order.
- Overlapping genes reuse checkpoint weights; new genes are initialized.


## Scenario 6: Freeze or update gene embeddings

```bash
# Freeze gene embeddings during finetune
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --update-gene-embedding false

# Update gene embeddings during finetune
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode finetune \
  --update-gene-embedding true
```

Notes:
- Updating embeddings can help new genes adapt in overlap mode.
- Freezing can stabilize training when data is limited.


## Scenario 7: Export gene embeddings

```bash
segger segment -i /path/to/data -o /path/to/out \
  --export-gene-embeddings \
  --gene-embeddings-filename gene_embeddings.parquet
```

Notes:
- Writes a parquet with `feature_name` plus embedding columns `emb_0..emb_n`.
- Useful for downstream analysis or reuse.


## Scenario 8: Combine checkpointing with output formats

```bash
segger segment -i /path/to/data -o /path/to/out \
  --checkpoint-path /path/to/model.ckpt \
  --checkpoint-mode predict \
  --output-format all
```

Notes:
- Produces segmentation parquet plus merged, spatialdata, and anndata outputs.


## Summary

- Use `checkpoint-mode predict` for inference-only.
- Use `checkpoint-mode finetune` for transfer learning.
- Use `vocab-mode overlap` when gene lists differ and provide `checkpoint-vocab`.
- Export gene embeddings when you need reusable gene features.
